# 3 · Problem 1 — does legal pretraining help?

BERT against Legal-BERT. They share an architecture, a parameter count, a context
window and a chunk width, and differ only in pretraining corpus, so any gap between
them is attributable to domain adaptation.

**Needs a GPU.** Roughly 30–50 minutes for both on a T4.


## Setup

Clone the repository and install. The data layer needs nothing beyond the standard
library, so this is only for the model code.


In [ ]:
!git clone -q https://github.com/ManasDasri/NNDL.git
%cd NNDL
!pip install -q -e . 'matplotlib>=3.8'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


### Prepare the corpus

Extracts answer spans from `CUAD_v1.json` and assigns document-level splits.
Chunking happens at training time, because each model needs a different window.


In [ ]:
!legal-risk-prepare --cuad_json data/CUAD_v1.json --out_dir data/processed


### The baseline first

The comparison needs a floor to read against, and the CNN trains in a couple of
minutes on a GPU. Without it the table below has nothing to say about whether
110M pretrained parameters are earning their cost.


In [ ]:
!legal-risk-train-cnn --epochs 12 --batch_size 32 --output_dir outputs/cnn


### Train both

Identical settings. The only difference is the checkpoint.


In [ ]:
!legal-risk-train-transformer --model bert       --epochs 3 --batch_size 8 --lr 2e-5 --output_dir outputs/bert


In [ ]:
!legal-risk-train-transformer --model legal-bert --epochs 3 --batch_size 8 --lr 2e-5 --output_dir outputs/legal_bert


### Evaluate on the same held-out contracts


In [ ]:
for run in ('outputs/bert', 'outputs/legal_bert'):
    !legal-risk-evaluate --run_dir {run} --split test --pooling max


### Read the per-class columns, not the macro score

The hypothesis is specific: Legal-BERT should gain most on labels carried by legal
vocabulary, and least on labels carried by ordinary English such as Insurance.
A uniform gain across all six would mean something other than domain knowledge is
responsible.


In [ ]:
import json, pathlib

# Only compare runs that actually exist, so a missing one narrows the table
# instead of crashing after an hour of training.
CANDIDATES = (('CNN', 'cnn'), ('BERT', 'bert'), ('Legal-BERT', 'legal_bert'), ('Longformer', 'longformer'))
runs = {}
for name, directory in CANDIDATES:
    path = pathlib.Path('outputs') / directory / 'metrics.json'
    if path.exists():
        runs[name] = json.load(open(path))
    else:
        print(f'skipping {name}: no {path}')

assert runs, 'no finished runs found'
labels = [r['label'] for r in next(iter(runs.values()))['test_at_tuned_thresholds']['per_class']]

print(f"\n{'label':32s}" + ''.join(f'{n:>13s}' for n in runs))
for i, label in enumerate(labels):
    f1 = {n: r['test_at_tuned_thresholds']['per_class'][i]['f1'] for n, r in runs.items()}
    row = f"{label:32s}" + ''.join(f'{f1[n]:13.3f}' for n in runs)
    if 'BERT' in f1 and 'Legal-BERT' in f1:
        row += f"   Δ {f1['Legal-BERT'] - f1['BERT']:+.3f}"
    print(row)

macro = {n: r['test_at_tuned_thresholds']['macro_f1'] for n, r in runs.items()}
row = f"\n{'macro F1':32s}" + ''.join(f'{macro[n]:13.3f}' for n in runs)
if 'BERT' in macro and 'Legal-BERT' in macro:
    row += f"   Δ {macro['Legal-BERT'] - macro['BERT']:+.3f}"
print(row)


### Before claiming a result

Run-to-run variance on identical settings is about **±0.03 macro F1**, from float
nondeterminism on GPU. A gap smaller than that is noise.

If the difference is under 0.03, rerun both across several seeds and report the
spread. The honest finding may be *"domain pretraining did not measurably help on
this task at this scale"* — which is a real result, and more interesting than a
margin inside the noise floor.


In [ ]:
# Several seeds, if the headline gap is small.
for seed in (1, 2, 3):
    for model, out in (('bert', 'bert'), ('legal-bert', 'legal_bert')):
        !legal-risk-train-transformer --model {model} --epochs 3 --seed {seed} --output_dir outputs/{out}_s{seed}
